# Tree-based Model Experiments for Healthcare Risk Prediction

This notebook documents and executes the model experiments for the project **A Comparative Study of Tree-based Models for Healthcare Risk Prediction: Cross-validation and SHAP-based Stability Analysis**.

The notebook focuses on the AI Engineer (Model) scope:

- **RQ1:** How do AdaBoost, XGBoost, and LightGBM compare in healthcare risk prediction across multiple datasets?
- **RQ2:** How stable is the predictive performance of these models across cross-validation folds?

**RQ3 (SHAP-based feature-ranking stability) is analyzed separately.**

Reusable implementation is kept in `src/`, while this notebook provides the experiment setup, execution, results, and interpretation.

## 1. Environment and imports

The experiment configuration is stored in `config.yaml`. Model creation, evaluation metrics, and the reusable experiment runner are imported from `src/`.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

# Make the notebook work whether it is launched from the repository root
# or from the notebooks/ directory.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "config.yaml"
RESULT_DIR = PROJECT_ROOT / "results" / "model_evaluation"

from src.experiment_config import load_experiment_config
from src.experiments.run_models import run_experiments

print(f"Project root: {PROJECT_ROOT}")
print(f"Config:       {CONFIG_PATH}")
print(f"Results:      {RESULT_DIR}")

Project root: D:\Cert AIO\Mod3\healthcare-risk-prediction
Config:       D:\Cert AIO\Mod3\healthcare-risk-prediction\config.yaml
Results:      D:\Cert AIO\Mod3\healthcare-risk-prediction\results\model_evaluation


## 2. Experiment setup

The same fixed configuration is used for all three datasets so that differences in performance reflect model and dataset behavior rather than separate per-dataset hyperparameter searches. Stratified 5-fold cross-validation is used with a fixed random seed, and PR-AUC is the primary ranking metric because some datasets are strongly class-imbalanced.

In [2]:
config, _ = load_experiment_config(CONFIG_PATH)
experiment = config["experiment"]

setup_table = pd.DataFrame([
    {"Setting": "Random seed", "Value": experiment["random_state"]},
    {"Setting": "CV strategy", "Value": f"Stratified {experiment['cv']['n_splits']}-fold CV"},
    {"Setting": "Shuffle", "Value": experiment["cv"]["shuffle"]},
    {"Setting": "Decision threshold", "Value": experiment["decision_threshold"]},
    {"Setting": "Primary metric", "Value": experiment["primary_metric"]},
    {"Setting": "Balanced sample weights", "Value": experiment["balance_training"]},
])

display(setup_table)

,Setting,Value
0,Random seed,42
1,CV strategy,Stratified 5-fold CV
2,Shuffle,True
3,Decision threshold,0.5
4,Primary metric,pr_auc
5,Balanced sample weights,True


### 2.1 Datasets

In [3]:
dataset_table = pd.DataFrame([
    {
        "Dataset key": key,
        "Dataset": value["name"],
        "Development file": value["train_path"],
        "Test file": value["test_path"],
        "Target": value["target_column"],
    }
    for key, value in config["datasets"].items()
])

display(dataset_table)

,Dataset key,Dataset,Development file,Test file,Target
0,dataset1,Personal Key Indicators of Heart Disease (2022),data/processed/dataset1/train.csv,data/processed/dataset1/test.csv,HadHeartAttack
1,dataset2,Heart Failure Prediction,data/processed/dataset2/train.csv,data/processed/dataset2/test.csv,HeartDisease
2,dataset3,Heart Disease Health Indicators (BRFSS 2015),data/processed/dataset3/train.csv,data/processed/dataset3/test.csv,HeartDiseaseorAttack


### 2.2 Models and hyperparameters

In [4]:
hyperparameter_rows = []
for model_key, model_cfg in config["models"].items():
    for parameter, value in model_cfg["params"].items():
        if isinstance(value, dict):
            value = ", ".join(f"{k}={v}" for k, v in value.items())
        hyperparameter_rows.append({
            "Model": model_cfg.get("display_name", model_key),
            "Parameter": parameter,
            "Value": value,
        })

hyperparameters = pd.DataFrame(hyperparameter_rows)
display(hyperparameters)

,Model,Parameter,Value
0,AdaBoost,estimator,max_depth=1
1,AdaBoost,n_estimators,200
2,AdaBoost,learning_rate,0.1
3,XGBoost,objective,binary:logistic
4,XGBoost,eval_metric,logloss
5,XGBoost,n_estimators,300
6,XGBoost,learning_rate,0.05
7,XGBoost,max_depth,4
8,XGBoost,min_child_weight,1
9,XGBoost,subsample,0.8


## 3. Execute the experiments

The cell below calls the reusable experiment runner in `src/experiments/run_models`. It performs the same workflow for every dataset/model pair:

1. Load the processed development and untouched test splits.
2. Run stratified 5-fold cross-validation on the development split.
3. Train a fresh model in every fold.
4. Compute ROC-AUC, PR-AUC, Recall, and F1 on each validation fold.
5. Refit the model on the full development split and evaluate once on the hold-out test set.
6. Save fold-level, summary, comparison, and metadata files to `results/model_evaluation/`.

`RUN_EXPERIMENTS=True` reproduces the full experiment. After a successful run, it can be changed to `False` when reopening the notebook simply to inspect previously saved results.

In [5]:
RUN_EXPERIMENTS = True

if RUN_EXPERIMENTS:
    output_paths = run_experiments(config_path=CONFIG_PATH)
    print("\nExperiment completed successfully.")
    for name, path in output_paths.items():
        print(f"{name:28s}: {path}")
else:
    print("Skipping execution and using existing files in:", RESULT_DIR)


=== Personal Key Indicators of Heart Disease (2022) (dataset1) ===
Development: (353653, 132) | Test: (88414, 132) | positive rate=0.0568/0.0568
  -> AdaBoost
    Fold 1/5 | roc_auc=0.8774, pr_auc=0.3969, recall=0.6853, f1=0.3890 | fit=107.96s
    Fold 2/5 | roc_auc=0.8684, pr_auc=0.3760, recall=0.6794, f1=0.3677 | fit=73.42s
    Fold 3/5 | roc_auc=0.8738, pr_auc=0.3910, recall=0.6914, f1=0.3725 | fit=73.72s
    Fold 4/5 | roc_auc=0.8669, pr_auc=0.3783, recall=0.6629, f1=0.3798 | fit=74.57s
    Fold 5/5 | roc_auc=0.8713, pr_auc=0.3845, recall=0.6781, f1=0.3733 | fit=74.56s
    Hold-out test | roc_auc=0.8695, pr_auc=0.3860, recall=0.6705, f1=0.3657 | fit=88.55s
  -> XGBoost
    Fold 1/5 | roc_auc=0.8922, pr_auc=0.4257, recall=0.7857, f1=0.3356 | fit=15.41s
    Fold 2/5 | roc_auc=0.8850, pr_auc=0.4089, recall=0.7742, f1=0.3344 | fit=5.46s
    Fold 3/5 | roc_auc=0.8890, pr_auc=0.4198, recall=0.7857, f1=0.3311 | fit=3.96s
    Fold 4/5 | roc_auc=0.8850, pr_auc=0.4049, recall=0.7700, f1=0.3

## 4. Load experiment outputs

In [6]:
fold_metrics = pd.read_csv(RESULT_DIR / "fold_metrics.csv")
cv_summary = pd.read_csv(RESULT_DIR / "cv_summary.csv")
test_metrics = pd.read_csv(RESULT_DIR / "test_metrics.csv")
model_comparison = pd.read_csv(RESULT_DIR / "model_comparison.csv")
overall_comparison = pd.read_csv(RESULT_DIR / "overall_model_comparison.csv")

print("Fold-level rows:", len(fold_metrics))
print("CV summary rows: ", len(cv_summary))
print("Test rows:       ", len(test_metrics))

Fold-level rows: 45
CV summary rows:  9
Test rows:        9


### 4.1 Dataset characteristics observed by the model pipeline

In [7]:
dataset_summary = (
    test_metrics[[
        "dataset_key", "dataset_name", "n_development", "n_test",
        "development_positive_rate", "test_positive_rate"
    ]]
    .drop_duplicates()
    .sort_values("dataset_key")
    .reset_index(drop=True)
)

dataset_summary["development_positive_rate"] = (100 * dataset_summary["development_positive_rate"]).round(2)
dataset_summary["test_positive_rate"] = (100 * dataset_summary["test_positive_rate"]).round(2)
dataset_summary = dataset_summary.rename(columns={
    "dataset_key": "Dataset key",
    "dataset_name": "Dataset",
    "n_development": "Development samples",
    "n_test": "Test samples",
    "development_positive_rate": "Development positive (%)",
    "test_positive_rate": "Test positive (%)",
})

display(dataset_summary)

,Dataset key,Dataset,Development samples,Test samples,Development positive (%),Test positive (%)
0,dataset1,Personal Key Indicators of Heart Disease (2022),353653,88414,5.68,5.68
1,dataset2,Heart Failure Prediction,734,184,55.31,55.43
2,dataset3,Heart Disease Health Indicators (BRFSS 2015),183824,45957,10.32,10.32


## 5. RQ1 - Comparison of predictive performance

For RQ1, the models are compared using the mean 5-fold CV metrics within each dataset. PR-AUC is used as the primary ranking metric, with ROC-AUC, Recall, and F1 reported as complementary measures.

In [8]:
rq1_columns = [
    "dataset_name", "model_name",
    "roc_auc_mean", "pr_auc_mean", "recall_mean", "f1_mean",
    "cv_rank"
]

rq1_table = model_comparison[rq1_columns].copy()
rq1_table = rq1_table.sort_values(["dataset_name", "cv_rank"])
for col in ["roc_auc_mean", "pr_auc_mean", "recall_mean", "f1_mean"]:
    rq1_table[col] = rq1_table[col].round(4)

display(rq1_table)

,dataset_name,model_name,roc_auc_mean,pr_auc_mean,recall_mean,f1_mean,cv_rank
6,Heart Disease Health Indicators (BRFSS 2015),LightGBM,0.8381,0.3801,0.7970,0.3825,1
7,Heart Disease Health Indicators (BRFSS 2015),XGBoost,0.8386,0.3800,0.8016,0.3814,2
8,Heart Disease Health Indicators (BRFSS 2015),AdaBoost,0.8324,0.3681,0.7743,0.3825,3
3,Heart Failure Prediction,AdaBoost,0.9241,0.9226,0.8449,0.8578,1
4,Heart Failure Prediction,XGBoost,0.9244,0.9214,0.8819,0.8785,2
5,Heart Failure Prediction,LightGBM,0.9241,0.9152,0.8769,0.8726,3
0,Personal Key Indicators of Heart Disease (2022),XGBoost,0.8872,0.4138,0.7782,0.3324,1
1,Personal Key Indicators of Heart Disease (2022),LightGBM,0.8873,0.4129,0.7771,0.3346,2
2,Personal Key Indicators of Heart Disease (2022),AdaBoost,0.8715,0.3853,0.6794,0.3765,3


### 5.1 Cross-dataset comparison

In [9]:
overall_cols = [
    "model_name", "mean_dataset_rank",
    "pr_auc_mean", "roc_auc_mean", "recall_mean", "f1_mean"
]

overall_table = overall_comparison[overall_cols].copy()
for col in overall_cols[1:]:
    overall_table[col] = overall_table[col].round(4)

display(overall_table)

,model_name,mean_dataset_rank,pr_auc_mean,roc_auc_mean,recall_mean,f1_mean
0,XGBoost,1.6667,0.5717,0.8834,0.8205,0.5308
1,LightGBM,2.0000,0.5694,0.8832,0.8170,0.5299
2,AdaBoost,2.3333,0.5587,0.8760,0.7662,0.5389


In [10]:
best_rank_row = overall_comparison.sort_values("mean_dataset_rank").iloc[0]
best_pr_row = overall_comparison.sort_values("pr_auc_mean", ascending=False).iloc[0]
best_roc_row = overall_comparison.sort_values("roc_auc_mean", ascending=False).iloc[0]
best_recall_row = overall_comparison.sort_values("recall_mean", ascending=False).iloc[0]
best_f1_row = overall_comparison.sort_values("f1_mean", ascending=False).iloc[0]

rq1_analysis = f"""
### RQ1 Analysis

- **{best_rank_row['model_name']}** achieved the best overall mean dataset rank ({best_rank_row['mean_dataset_rank']:.2f}).
- The highest mean **PR-AUC** was achieved by **{best_pr_row['model_name']}** ({best_pr_row['pr_auc_mean']:.4f}).
- The highest mean **ROC-AUC** was achieved by **{best_roc_row['model_name']}** ({best_roc_row['roc_auc_mean']:.4f}).
- The highest mean **Recall** was achieved by **{best_recall_row['model_name']}** ({best_recall_row['recall_mean']:.4f}).
- The highest mean **F1** was achieved by **{best_f1_row['model_name']}** ({best_f1_row['f1_mean']:.4f}).

These results should be interpreted as a multi-metric comparison rather than as evidence that one model dominates every criterion. In particular, ROC-AUC and PR-AUC measure ranking/discrimination across thresholds, whereas Recall and F1 depend on the fixed decision threshold used in the experiment.
"""

display(Markdown(rq1_analysis))


### RQ1 Analysis

- **XGBoost** achieved the best overall mean dataset rank (1.67).
- The highest mean **PR-AUC** was achieved by **XGBoost** (0.5717).
- The highest mean **ROC-AUC** was achieved by **XGBoost** (0.8834).
- The highest mean **Recall** was achieved by **XGBoost** (0.8205).
- The highest mean **F1** was achieved by **AdaBoost** (0.5389).

These results should be interpreted as a multi-metric comparison rather than as evidence that one model dominates every criterion. In particular, ROC-AUC and PR-AUC measure ranking/discrimination across thresholds, whereas Recall and F1 depend on the fixed decision threshold used in the experiment.


### 5.2 Final hold-out test performance

In [11]:
test_view = test_metrics[[
    "dataset_name", "model_name", "roc_auc", "pr_auc", "recall", "f1"
]].copy()
for col in ["roc_auc", "pr_auc", "recall", "f1"]:
    test_view[col] = test_view[col].round(4)

display(test_view.sort_values(["dataset_name", "model_name"]))

,dataset_name,model_name,roc_auc,pr_auc,recall,f1
6,Heart Disease Health Indicators (BRFSS 2015),AdaBoost,0.8326,0.3672,0.7674,0.3788
8,Heart Disease Health Indicators (BRFSS 2015),LightGBM,0.8385,0.3763,0.7925,0.3789
7,Heart Disease Health Indicators (BRFSS 2015),XGBoost,0.8384,0.3756,0.7963,0.3773
3,Heart Failure Prediction,AdaBoost,0.9256,0.9394,0.8529,0.8832
5,Heart Failure Prediction,LightGBM,0.9189,0.9166,0.8431,0.8643
4,Heart Failure Prediction,XGBoost,0.9224,0.9216,0.8333,0.8586
0,Personal Key Indicators of Heart Disease (2022),AdaBoost,0.8695,0.3860,0.6705,0.3657
2,Personal Key Indicators of Heart Disease (2022),LightGBM,0.8850,0.4187,0.7694,0.3259
1,Personal Key Indicators of Heart Disease (2022),XGBoost,0.8850,0.4169,0.7712,0.3255


The hold-out test set is not used during cross-validation. Similar CV and hold-out results provide an additional check that the reported comparison is not driven only by a favorable validation fold.

## 6. RQ2 - Stability across cross-validation folds

RQ2 examines how sensitive each model's predictive performance is to the specific training/validation partition. Stability is summarized using the standard deviation and range across the five folds. Smaller values indicate more consistent fold-to-fold performance.

In [12]:
stability_cols = [
    "dataset_name", "model_name",
    "roc_auc_mean", "roc_auc_std", "roc_auc_range",
    "pr_auc_mean", "pr_auc_std", "pr_auc_range",
    "recall_mean", "recall_std", "recall_range",
    "f1_mean", "f1_std", "f1_range",
]

stability_table = cv_summary[stability_cols].copy()
metric_cols = [c for c in stability_table.columns if c not in ["dataset_name", "model_name"]]
stability_table[metric_cols] = stability_table[metric_cols].round(4)

display(stability_table.sort_values(["dataset_name", "model_name"]))

,dataset_name,model_name,roc_auc_mean,roc_auc_std,roc_auc_range,pr_auc_mean,pr_auc_std,pr_auc_range,recall_mean,recall_std,recall_range,f1_mean,f1_std,f1_range
6,Heart Disease Health Indicators (BRFSS 2015),AdaBoost,0.8324,0.0029,0.0067,0.3681,0.0074,0.0183,0.7743,0.0105,0.0264,0.3825,0.0027,0.0066
7,Heart Disease Health Indicators (BRFSS 2015),LightGBM,0.8381,0.0027,0.0055,0.3801,0.0056,0.0128,0.7970,0.0053,0.0134,0.3825,0.0021,0.0047
8,Heart Disease Health Indicators (BRFSS 2015),XGBoost,0.8386,0.0027,0.0054,0.3800,0.0053,0.0131,0.8016,0.0048,0.0121,0.3814,0.0020,0.0041
3,Heart Failure Prediction,AdaBoost,0.9241,0.0418,0.1126,0.9226,0.0435,0.1179,0.8449,0.0294,0.0741,0.8578,0.0190,0.0437
4,Heart Failure Prediction,LightGBM,0.9241,0.0338,0.0845,0.9152,0.0355,0.0865,0.8769,0.0366,0.0741,0.8726,0.0256,0.0677
5,Heart Failure Prediction,XGBoost,0.9244,0.0322,0.0789,0.9214,0.0320,0.0761,0.8819,0.0381,0.0968,0.8785,0.0248,0.0636
0,Personal Key Indicators of Heart Disease (2022),AdaBoost,0.8715,0.0042,0.0104,0.3853,0.0087,0.0209,0.6794,0.0106,0.0285,0.3765,0.0082,0.0213
1,Personal Key Indicators of Heart Disease (2022),LightGBM,0.8873,0.0033,0.0073,0.4129,0.0088,0.0212,0.7771,0.0084,0.0194,0.3346,0.0031,0.0075
2,Personal Key Indicators of Heart Disease (2022),XGBoost,0.8872,0.0033,0.0074,0.4138,0.0086,0.0208,0.7782,0.0071,0.0157,0.3324,0.0025,0.0062


### 6.1 Fold-level ROC-AUC values

In [13]:
roc_fold_table = fold_metrics.pivot_table(
    index=["dataset_name", "model_name"],
    columns="fold",
    values="roc_auc"
).round(4)
roc_fold_table.columns = [f"Fold {int(c)}" for c in roc_fold_table.columns]

display(roc_fold_table)

Fold 1  Fold 2  \
dataset_name                                    model_name                   
Heart Disease Health Indicators (BRFSS 2015)    AdaBoost    0.8345  0.8356   
                                                LightGBM    0.8403  0.8394   
                                                XGBoost     0.8407  0.8403   
Heart Failure Prediction                        AdaBoost    0.9271  0.8547   
                                                LightGBM    0.9285  0.8651   
                                                XGBoost     0.9340  0.8676   
Personal Key Indicators of Heart Disease (2022) AdaBoost    0.8774  0.8684   
                                                LightGBM    0.8923  0.8852   
                                                XGBoost     0.8922  0.8850   

                                                            Fold 3  Fold 4  \
dataset_name                                    model_name                   
Heart Disease Health Indicators (BRFSS 2015)    AdaBoost    0.8298  0.8289   
                                                LightGBM    0.8356  0.8349   
                                                XGBoost     0.8360  0.8353   
Heart Failure Prediction                        AdaBoost    0.9401  0.9673   
                                                LightGBM    0.9383  0.9497   
                                                XGBoost     0.9394  0.9465   
Personal Key Indicators of Heart Disease (2022) AdaBoost    0.8738  0.8669   
                                                LightGBM    0.8892  0.8850   
                                                XGBoost     0.8890  0.8850   

                                                            Fold 5  
dataset_name                                    model_name          
Heart Disease Health Indicators (BRFSS 2015)    AdaBoost    0.8332  
                                                LightGBM    0.8403  
                                                XGBoost     0.8406  
Heart Failure Prediction                        AdaBoost    0.9311  
                                                LightGBM    0.9390  
                                                XGBoost     0.9347  
Personal Key Indicators of Heart Disease (2022) AdaBoost    0.8713  
                                                LightGBM    0.8850  
                                                XGBoost     0.8848

In [14]:
# Identify the most and least stable dataset/model combinations by ROC-AUC standard deviation.
stability_rank = cv_summary[["dataset_name", "model_name", "roc_auc_std", "pr_auc_std"]].copy()
most_stable = stability_rank.sort_values("roc_auc_std").iloc[0]
least_stable = stability_rank.sort_values("roc_auc_std", ascending=False).iloc[0]

# Dataset-level average variability helps compare the three datasets.
dataset_variability = (
    cv_summary.groupby("dataset_name", as_index=False)
    .agg(
        mean_roc_auc_std=("roc_auc_std", "mean"),
        mean_pr_auc_std=("pr_auc_std", "mean"),
    )
    .sort_values("mean_roc_auc_std")
)

display(dataset_variability.round(4))

rq2_analysis = f"""
### RQ2 Analysis

- The most stable model/dataset combination by ROC-AUC standard deviation is **{most_stable['model_name']}** on **{most_stable['dataset_name']}** (std = {most_stable['roc_auc_std']:.4f}).
- The largest ROC-AUC fold variation occurs for **{least_stable['model_name']}** on **{least_stable['dataset_name']}** (std = {least_stable['roc_auc_std']:.4f}).
- Stability should be interpreted together with dataset size and class distribution. A smaller dataset may be more sensitive to which observations are assigned to each fold, although sample size is not necessarily the only explanation for variation.

Overall, the fold-level statistics make it possible to distinguish models that achieve strong average performance from models whose results are highly sensitive to a particular cross-validation split.
"""

display(Markdown(rq2_analysis))

,dataset_name,mean_roc_auc_std,mean_pr_auc_std
0,Heart Disease Health Indicators (BRFSS 2015),0.0028,0.0061
2,Personal Key Indicators of Heart Disease (2022),0.0036,0.0087
1,Heart Failure Prediction,0.0359,0.0370



### RQ2 Analysis

- The most stable model/dataset combination by ROC-AUC standard deviation is **LightGBM** on **Heart Disease Health Indicators (BRFSS 2015)** (std = 0.0027).
- The largest ROC-AUC fold variation occurs for **AdaBoost** on **Heart Failure Prediction** (std = 0.0418).
- Stability should be interpreted together with dataset size and class distribution. A smaller dataset may be more sensitive to which observations are assigned to each fold, although sample size is not necessarily the only explanation for variation.

Overall, the fold-level statistics make it possible to distinguish models that achieve strong average performance from models whose results are highly sensitive to a particular cross-validation split.


## 7. Experiment summary

This notebook provides the reproducible evidence for the model-comparison component of the study:

- all three models use the same processed inputs and fixed experimental configuration;
- RQ1 is evaluated with ROC-AUC, PR-AUC, Recall, F1, within-dataset ranks, and cross-dataset averages;
- RQ2 is evaluated using fold-level results and variability statistics such as standard deviation and range;
- final hold-out test results are reported separately from cross-validation;
- reusable implementation remains in `src/`, while this notebook records experiment execution and interpretation.

The SHAP-based feature-ranking stability analysis for **RQ3** is outside the model-comparison scope of this notebook and can be integrated separately.